# Student Performance Prediction — EDA and model review

This notebook documents the supplied dataset, basic quality checks, descriptive analysis, and the model comparison produced by `python -m src.train`. It uses paths relative to the repository root.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "Student_Performance.csv").exists():
    ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "Student_Performance.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## Data quality

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.rename("dtype").to_frame())
display(df.isna().sum().rename("missing").to_frame())
print("Exact duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## Distributions and associations

In [ ]:
numeric = df.select_dtypes(include="number")
display(numeric.corr()["Performance Index"].sort_values(ascending=False).to_frame("Pearson correlation"))
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["Hours Studied", "Previous Scores", "Sleep Hours", "Sample Question Papers Practiced"]):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="Previous Scores", y="Performance Index", alpha=.25, ax=axes[0])
sns.boxplot(data=df, x="Extracurricular Activities", y="Performance Index", ax=axes[1])
plt.tight_layout()

## Duplicate and outlier review

Exact duplicates are removed before the train/test split in the training workflow. The following Tukey fences are descriptive flags only; the project does not automatically remove outliers.

In [ ]:
dedup = df.drop_duplicates()
print(f"Rows: {len(df):,}; exact duplicates: {df.duplicated().sum():,}; unique rows: {len(dedup):,}")
for col in numeric.columns:
    q1, q3 = df[col].quantile([.25, .75])
    iqr = q3 - q1
    flags = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col}: {flags} Tukey-flagged values")

## Model comparison

Run `python -m src.train` from the repository root to refresh the cross-validation and holdout metrics.

In [ ]:
metrics_path = ROOT / "reports" / "model_comparison.csv"
if metrics_path.exists():
    display(pd.read_csv(metrics_path).sort_values("cv_rmse_mean"))
else:
    print("No model report found yet. Run: python -m src.train")

## Interpretation and limitations

Previous Scores is strongly associated with Performance Index in this dataset; the other numeric features have weaker linear correlations. These associations do not establish causality. The archive does not provide provenance or target-generation documentation, and no external validation set is available. Therefore the high holdout score should be interpreted as within-dataset performance only.